# START : Liz Choi

# Objective
The goal of this notebook is to retrain and evaluate a **Random Forest Regressor** for predicting residential property closing prices.
We use the newly engineered features (e.g., Bed_Bath_Ratio, Property_Age, Months_From_Dec_2025, Sqft_Per_Bedroom, Lot_Utilization) to improve predictive performance.

## Import Libraries and Load Data
We load the feature-engineered training and test datasets saved from the previous feature engineering step.

In [60]:
import pandas as pd
import numpy as np

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_absolute_percentage_error, mean_squared_error

train = pd.read_csv("train_cleaned_fe.csv")
test  = pd.read_csv("test_cleaned_fe.csv")

## Log Transformation of Target Variable
We model the **log-transformed ClosePrice** to reduce skew and make errors more stable across price ranges.

In [62]:
train["LogClosePrice"] = np.log(train["ClosePrice"])
test["LogClosePrice"]  = np.log(test["ClosePrice"])

## Select Features for Random Forest
We define a numeric feature set (same style as the Decision Tree notebook).
All selected features are converted to numeric format.

In [64]:
rf_features = [
    "LivingArea",
    "BathroomsTotalInteger",
    "BedroomsTotal",
    "PPSF",
    "Bed_Bath_Ratio",
    "Property_Age",
    "Months_From_Dec_2025",
    "Sqft_Per_Bedroom",
    "Lot_Utilization"
]

for col in rf_features:
    train[col] = pd.to_numeric(train[col], errors="coerce")
    test[col]  = pd.to_numeric(test[col], errors="coerce")

## Remove NaN / inf
Random forests cannot handle NaN/inf values directly.
We replace inf with NaN and drop rows that are missing required features or the target.

In [66]:
for df in [train, test]:
    df.replace([np.inf, -np.inf], np.nan, inplace=True)

train = train.dropna(subset=rf_features + ["LogClosePrice"])
test  = test.dropna(subset=rf_features + ["LogClosePrice"])

## Prepare Feature Matrices and Target Variables
- **X_train / X_test**: input feature matrices
- **y_train / y_test**: log-transformed target (`LogClosePrice`)
Features are cast to **float32** to improve efficiency.

In [68]:
X_train = train[rf_features].astype(np.float32)
X_test  = test[rf_features].astype(np.float32)

y_train = train["LogClosePrice"]
y_test  = test["LogClosePrice"]

## Train Random Forest Regressor
A Random Forest is an ensemble of many decision trees.
Key parameters control complexity and help reduce overfitting:
- `n_estimators`: number of trees
- `max_depth`: max tree depth
- `min_samples_leaf`: minimum samples per leaf

In [70]:
rf_model = RandomForestRegressor(
    n_estimators=600,
    max_depth=18,
    min_samples_leaf=20,
    random_state=42,
    n_jobs=-1
)

rf_model.fit(X_train, y_train)

RandomForestRegressor(max_depth=18, min_samples_leaf=20, n_estimators=600,
                      n_jobs=-1, random_state=42)

## Generate Predictions
We generate predictions on both train and test sets.
Predictions are in **log scale** and later converted back to dollar scale for error metrics.

In [71]:
train_pred = rf_model.predict(X_train)
test_pred  = rf_model.predict(X_test)

## Model Performance Evaluation
We report:
- **R²** on log scale
- **RMSE** on dollar scale
- **MAPE** and **MdAPE** on dollar scale

In [72]:
train_r2 = r2_score(y_train, train_pred)
test_r2  = r2_score(y_test, test_pred)

y_train_d = np.exp(y_train)
y_test_d  = np.exp(y_test)

train_pred_d = np.exp(train_pred)
test_pred_d  = np.exp(test_pred)

train_rmse = mean_squared_error(y_train_d, train_pred_d, squared=False)
test_rmse  = mean_squared_error(y_test_d, test_pred_d, squared=False)

train_mape = mean_absolute_percentage_error(y_train_d, train_pred_d) * 100
test_mape  = mean_absolute_percentage_error(y_test_d, test_pred_d) * 100

train_mdape = np.median(np.abs((y_train_d - train_pred_d) / y_train_d)) * 100
test_mdape  = np.median(np.abs((y_test_d - test_pred_d) / y_test_d)) * 100

print("Random Forest Performance")
print(f"Train R2 (log): {train_r2:.4f} | Test R2 (log): {test_r2:.4f}")
print(f"Train RMSE ($): {train_rmse:,.2f} | Test RMSE ($): {test_rmse:,.2f}")
print(f"Train MAPE (%): {train_mape:.2f} | Test MAPE (%): {test_mape:.2f}")
print(f"Train MdAPE(%): {train_mdape:.2f} | Test MdAPE(%): {test_mdape:.2f}")

Random Forest Performance
Train R2 (log): 0.9991 | Test R2 (log): 0.9985
Train RMSE ($): 64,397.26 | Test RMSE ($): 95,171.73
Train MAPE (%): 0.55 | Test MAPE (%): 0.63
Train MdAPE(%): 0.26 | Test MdAPE(%): 0.30


/opt/anaconda3/lib/python3.12/site-packages/sklearn/metrics/_regression.py:492: FutureWarning: 'squared' is deprecated in version 1.4 and will be removed in 1.6. To calculate the root mean squared error, use the function'root_mean_squared_error'.
  warnings.warn(
/opt/anaconda3/lib/python3.12/site-packages/sklearn/metrics/_regression.py:492: FutureWarning: 'squared' is deprecated in version 1.4 and will be removed in 1.6. To calculate the root mean squared error, use the function'root_mean_squared_error'.
  warnings.warn(


## Feature Importance
Random Forest provides feature importance scores which help interpret which engineered features contribute most.

In [73]:
importances = pd.DataFrame({
    "Feature": rf_features,
    "Importance": rf_model.feature_importances_
}).sort_values("Importance", ascending=False)

importances

,Feature,Importance
3,PPSF,5.907522e-01
0,LivingArea,4.092433e-01
7,Sqft_Per_Bedroom,3.281842e-06
8,Lot_Utilization,4.176657e-07
5,Property_Age,2.280924e-07
1,BathroomsTotalInteger,2.134192e-07
2,BedroomsTotal,1.762072e-07
6,Months_From_Dec_2025,1.379054e-07
4,Bed_Bath_Ratio,4.404224e-08


# END : Liz Choi 